In [0]:
%sql
WITH
-- =====================================================================
-- MONTH PARAMETER
-- Set month_start to the first day of the reporting month. month_end
-- (EOM) and prior_month_end are derived from it.
--   - Event/flow filters (signups, exits, upgrades, payments, gift
--     purchases/redemptions) now scan the full month_start..month_end
--     range instead of a single day.
--   - Status/snapshot checks (active balance, lapsed, dunning, RFM
--     segment) are anchored to month_end vs prior_month_end instead
--     of today vs yesterday.
-- =====================================================================
report_period AS (
    SELECT
        DATE'2025-08-11'              AS month_start,
        DATE'2025-08-31'              AS month_end,
        date_sub(DATE'2025-08-11', 1)   AS prior_month_end
),

first_timers AS (
    SELECT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    GROUP BY cc_user_profile_id
    HAVING COUNT(*) = 1
),

-- MONTHLY (converted from scalar counts to member lists — same filters, untouched)
m_total_signups AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND plan_id = 'monthly1'
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
m_first_time AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    INNER JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'monthly1'
      AND e.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
m_returning AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'monthly1'
      AND e.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
m_returning_breakdown AS (
    SELECT DISTINCT oe.cc_user_profile_id, oe.prev_plan_id
    FROM (
        SELECT cc_user_profile_id, plan_id, reporting_datetime,
               LAG(plan_id) OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime) AS prev_plan_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    ) AS oe
    LEFT ANTI JOIN first_timers AS ft ON oe.cc_user_profile_id = ft.cc_user_profile_id
    WHERE oe.plan_id = 'monthly1'
      AND oe.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
m_exits AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND cc_user_profile_id IS NOT NULL
      AND changed IS NOT TRUE
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
-- ANNUAL (converted from scalar counts to member lists — same filters, untouched)
-- upgrades_took_effect: the monthly1 term formally ending as part of an upgrade completing.
-- Some completions log this moment as a 'subscription_created' (annual1) event with a real
-- reporting_datetime instead of 'subscription_started', which lets them leak into the four
-- 'created' queries below as if they were a brand-new annual signup. Defined here so each of
-- those can exclude it.
upgrades_took_effect AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_total_signups AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_first_time AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    INNER JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_returning AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_returning_breakdown AS (
    SELECT DISTINCT oe.cc_user_profile_id, oe.prev_plan_id
    FROM (
        SELECT cc_user_profile_id, plan_id, reporting_datetime,
               LAG(plan_id) OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime) AS prev_plan_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    ) AS oe
    LEFT ANTI JOIN first_timers AS ft ON oe.cc_user_profile_id = ft.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON oe.cc_user_profile_id = ute.cc_user_profile_id
    WHERE oe.plan_id = 'annual1'
      AND oe.reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_upgrades AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_changed' AND changed = TRUE AND changed_from != changed_to
      AND changed_to = 'annual1'
      AND CAST(from_unixtime(record_occurred_at) AS DATE) BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
a_exits AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'annual1' AND cc_user_profile_id IS NOT NULL
      AND changed IS NOT TRUE
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),

-- annual_renewed: a payment on annual1 during the month lining up with a member's own
-- ~365-day anniversary (from their earliest dated signup or upgrade completion),
-- excluding this period's signups/exits/upgrade-completions so it can never
-- overlap with those buckets. Anchored to each payment's own date (not a fixed
-- day), since payments are now scanned across the whole month.
annual_direct_signup_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_upgrade_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_starts AS (
    SELECT cc_user_profile_id, MIN(start_date) AS start_date
    FROM (
        SELECT * FROM annual_direct_signup_starts
        UNION ALL
        SELECT * FROM annual_upgrade_starts
    )
    GROUP BY cc_user_profile_id
),
annual_payments_period AS (
    SELECT DISTINCT cc_user_profile_id, reporting_datetime AS payment_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
      AND reporting_datetime BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
annual_renewed AS (
    SELECT DISTINCT p.cc_user_profile_id
    FROM annual_payments_period AS p
    INNER JOIN annual_starts AS s ON p.cc_user_profile_id = s.cc_user_profile_id
    LEFT ANTI JOIN a_total_signups AS sg ON sg.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN a_exits AS ex ON ex.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON ute.cc_user_profile_id = p.cc_user_profile_id
    WHERE DATEDIFF(p.payment_date, s.start_date) >= 360
      AND (MOD(DATEDIFF(p.payment_date, s.start_date), 365) <= 5 OR MOD(DATEDIFF(p.payment_date, s.start_date), 365) >= 360)
),

a_prior_eom_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT prior_month_end FROM report_period), 'yyyyMMdd')
      AND plan_id = 'annual1'
),
a_eom_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT month_end FROM report_period), 'yyyyMMdd')
      AND plan_id = 'annual1'
),
a_last_payment AS (
    SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
           ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'annual1'
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
a_last_succeeded AS (
    SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
    GROUP BY cc_user_profile_id
),
a_eom_dunning AS (
    SELECT DISTINCT lp.cc_user_profile_id
    FROM a_last_payment AS lp
    LEFT JOIN a_last_succeeded AS ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
    WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
      AND lp.reporting_datetime >= DATE_SUB((SELECT month_end FROM report_period), 14)
      AND lp.reporting_datetime <= (SELECT month_end FROM report_period)
      AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
      AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
),
a_eom_active_or_dunning AS (
    SELECT cc_user_profile_id FROM a_eom_active
    UNION
    SELECT cc_user_profile_id FROM a_eom_dunning
),
a_any_cancellation_to_date AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'annual1' AND changed IS NOT TRUE
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
a_lapsed AS (
    SELECT pa.cc_user_profile_id
    FROM a_prior_eom_active AS pa
    LEFT ANTI JOIN a_eom_active_or_dunning AS ta ON pa.cc_user_profile_id = ta.cc_user_profile_id
    LEFT ANTI JOIN a_any_cancellation_to_date AS ac ON pa.cc_user_profile_id = ac.cc_user_profile_id
),

-- GIFT
g_purchasers AS (
    SELECT DISTINCT purchaser_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE CAST(from_utc_timestamp(CAST(purchase_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
g_redeemers AS (
    SELECT DISTINCT redeemer_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE is_redeemed = TRUE
      AND CAST(from_utc_timestamp(CAST(redemption_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN (SELECT month_start FROM report_period) AND (SELECT month_end FROM report_period)
),
prior_eom_snapshot AS (
    SELECT DISTINCT cc_user_profile_id, plan_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT prior_month_end FROM report_period), 'yyyyMMdd')
),
-- gift purchaser/redeemer breakdowns pulled out as member lists
gift_purchasers_net_new AS (
    SELECT t.purchaser_id AS cc_user_profile_id
    FROM g_purchasers t
    LEFT ANTI JOIN prior_eom_snapshot p ON t.purchaser_id = p.cc_user_profile_id
),
gift_purchasers_returning AS (
    SELECT t.purchaser_id AS cc_user_profile_id, p.plan_id
    FROM g_purchasers t
    INNER JOIN prior_eom_snapshot p ON t.purchaser_id = p.cc_user_profile_id
),
gift_redeemers_net_new AS (
    SELECT t.redeemer_id AS cc_user_profile_id
    FROM g_redeemers t
    LEFT ANTI JOIN prior_eom_snapshot p ON t.redeemer_id = p.cc_user_profile_id
),
gift_redeemers_returning AS (
    SELECT t.redeemer_id AS cc_user_profile_id, p.plan_id
    FROM g_redeemers t
    INNER JOIN prior_eom_snapshot p ON t.redeemer_id = p.cc_user_profile_id
),
g_lapsed_prior_eom_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT prior_month_end FROM report_period), 'yyyyMMdd')
      AND plan_id = 'gift-plan'
      AND renewal_status IN ('Active','Active_Non_Renewing','Active_Annual_Upgrade')
),
g_lapsed_eom_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT month_end FROM report_period), 'yyyyMMdd')
      AND plan_id = 'gift-plan'
      AND renewal_status IN ('Active','Active_Non_Renewing','Active_Annual_Upgrade')
),
g_any_cancellation_to_date AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'gift-plan' AND changed IS NOT TRUE
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
g_lapsed AS (
    SELECT pa.cc_user_profile_id
    FROM g_lapsed_prior_eom_active AS pa
    LEFT ANTI JOIN g_lapsed_eom_active AS ta ON pa.cc_user_profile_id = ta.cc_user_profile_id
    LEFT ANTI JOIN g_any_cancellation_to_date AS ac ON pa.cc_user_profile_id = ac.cc_user_profile_id
),

-- ACTIVE BALANCE (status as of EOM) — same event term-window logic as the
-- verified daily script, anchored to month_end instead of a literal date.
bal_monthly_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_annual_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND plan_id = 'annual1'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_gift_active AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND plan_id = 'gift-plan'
      AND (SELECT month_end FROM report_period) BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
),
bal_last_payment AS (
    SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
           ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'monthly1'
      AND reporting_datetime <= (SELECT month_end FROM report_period)
),
bal_last_succeeded AS (
    SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
    GROUP BY cc_user_profile_id
),
bal_monthly_dunning AS (
    SELECT DISTINCT lp.cc_user_profile_id
    FROM bal_last_payment AS lp
    LEFT JOIN bal_last_succeeded AS ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
    WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
      AND lp.reporting_datetime >= DATE_SUB((SELECT month_end FROM report_period), 15)
      AND lp.reporting_datetime <= (SELECT month_end FROM report_period)
      AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
      AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
),

-- =====================================================================
-- RFM LOOKUP LAYER
-- Simplified: no fallback union (all_members_seen removed) and no
-- duplicate 'Top 3 RFM' row — both are now derived by subtraction/sum
-- inside each *_by_rfm CTE below instead of being baked into the map.
-- Every member gets exactly one row here (Dormant via COALESCE covers
-- both an explicit 'Dormant' RFM segment and a null/no-match segment).
-- =====================================================================
cde_lookup_eom AS (
    SELECT DISTINCT cc_user_profile_id, cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT month_end FROM report_period), 'yyyyMMdd')
),
-- Fallback for members missing from the EOM snapshot (e.g. lapsed as of
-- EOM, so already dropped) — use their most recent prior (prior-EOM)
-- cde_id instead of letting them fall through to Dormant by default.
cde_lookup_prior_eom AS (
    SELECT DISTINCT cc_user_profile_id, cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = date_format((SELECT prior_month_end FROM report_period), 'yyyyMMdd')
),
cde_lookup AS (
    SELECT cc_user_profile_id, cde_id FROM cde_lookup_eom
    UNION ALL
    SELECT p.cc_user_profile_id, p.cde_id
    FROM cde_lookup_prior_eom p
    LEFT ANTI JOIN cde_lookup_eom t ON t.cc_user_profile_id = p.cc_user_profile_id
),
rfm_snapshot AS (
    SELECT CDE_ID, Segment
    FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model
    WHERE SnapshotDateId = date_format((SELECT month_end FROM report_period), 'yyyyMMdd')
),
member_segment AS (
    -- LEFT JOIN so a member with no RFM match still gets a row (Segment = NULL -> Dormant)
    SELECT cl.cc_user_profile_id, rs.Segment
    FROM cde_lookup cl
    LEFT JOIN rfm_snapshot rs ON CAST(cl.cde_id AS INT) = rs.CDE_ID
),
member_category_map AS (
    SELECT cc_user_profile_id, COALESCE(Segment, 'Dormant') AS rfm_category
    FROM member_segment
),
categories AS (
    SELECT 'All' AS rfm_category
    UNION ALL SELECT 'Top 3 RFM'
    UNION ALL SELECT 'Dormant'
    UNION ALL SELECT 'Champions'
    UNION ALL SELECT 'High Value Lapsed'
    UNION ALL SELECT 'Value Conscious Loyals'
    UNION ALL SELECT 'At Risk'
    UNION ALL SELECT 'Potential Loyalist'
    UNION ALL SELECT 'Promising'
    UNION ALL SELECT 'Fleeting'
    UNION ALL SELECT 'New'
),

-- =====================================================================
-- PER-METRIC RFM BREAKOUTS
-- Each metric now scans its source list against member_category_map
-- exactly once (the "_cats" CTE, non-Dormant real categories + 'All').
-- 'Top 3 RFM' and 'Dormant' are then derived from that same small
-- result set by SUM/subtraction — no extra scan of the source list,
-- and no need for the old 24-way all_members_seen fallback: any member
-- missing from member_category_map entirely is automatically swept
-- into Dormant via All - sum(named categories).
-- =====================================================================
monthly_signups_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_total_signups
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_total_signups xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_signups_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_signups_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_signups_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_signups_cats
),

monthly_first_time_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_first_time
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_first_time xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_first_time_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_first_time_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_first_time_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_first_time_cats
),

monthly_returning_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_returning
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_returning_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_returning_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_returning_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_returning_cats
),

monthly_returning_from_monthly_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_returning_breakdown WHERE prev_plan_id = 'monthly1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'monthly1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_returning_from_monthly_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_returning_from_monthly_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_returning_from_monthly_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_returning_from_monthly_cats
),

monthly_returning_from_annual_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_returning_breakdown WHERE prev_plan_id = 'annual1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'annual1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_returning_from_annual_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_returning_from_annual_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_returning_from_annual_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_returning_from_annual_cats
),

monthly_returning_from_gift_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_returning_breakdown WHERE prev_plan_id = 'gift-plan'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'gift-plan' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_returning_from_gift_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_returning_from_gift_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_returning_from_gift_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_returning_from_gift_cats
),

monthly_exits_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM m_exits
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM m_exits xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_exits_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_exits_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_exits_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_exits_cats
),

annual_signups_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_total_signups
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_total_signups xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_signups_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_signups_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_signups_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_signups_cats
),

annual_first_time_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_first_time
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_first_time xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_first_time_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_first_time_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_first_time_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_first_time_cats
),

annual_returning_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_returning
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_returning_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_returning_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_returning_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_returning_cats
),

annual_returning_from_monthly_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_returning_breakdown WHERE prev_plan_id = 'monthly1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'monthly1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_returning_from_monthly_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_returning_from_monthly_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_returning_from_monthly_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_returning_from_monthly_cats
),

annual_returning_from_annual_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_returning_breakdown WHERE prev_plan_id = 'annual1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'annual1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_returning_from_annual_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_returning_from_annual_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_returning_from_annual_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_returning_from_annual_cats
),

annual_returning_from_gift_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_returning_breakdown WHERE prev_plan_id = 'gift-plan'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_returning_breakdown xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.prev_plan_id = 'gift-plan' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_returning_from_gift_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_returning_from_gift_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_returning_from_gift_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_returning_from_gift_cats
),

upgrades_initiated_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_upgrades
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_upgrades xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
upgrades_initiated_by_rfm AS (
    SELECT rfm_category, cnt FROM upgrades_initiated_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM upgrades_initiated_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM upgrades_initiated_cats
),

upgrades_took_effect_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM upgrades_took_effect
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM upgrades_took_effect xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
upgrades_took_effect_by_rfm AS (
    SELECT rfm_category, cnt FROM upgrades_took_effect_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM upgrades_took_effect_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM upgrades_took_effect_cats
),

annual_exits_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_exits
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_exits xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_exits_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_exits_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_exits_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_exits_cats
),

annual_renewed_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM annual_renewed
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM annual_renewed xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_renewed_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_renewed_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_renewed_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_renewed_cats
),

annual_dunning_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_eom_dunning
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_eom_dunning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_dunning_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_dunning_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_dunning_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_dunning_cats
),

annual_lapsed_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM a_lapsed
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM a_lapsed xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_lapsed_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_lapsed_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_lapsed_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_lapsed_cats
),

gift_total_purchasers_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT purchaser_id) AS cnt FROM g_purchasers
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.purchaser_id)
    FROM g_purchasers xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.purchaser_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_total_purchasers_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_total_purchasers_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_total_purchasers_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_total_purchasers_cats
),

gift_purchasers_net_new_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_purchasers_net_new
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_purchasers_net_new xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_purchasers_net_new_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_purchasers_net_new_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_purchasers_net_new_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_purchasers_net_new_cats
),

gift_purchasers_returning_from_monthly_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_purchasers_returning WHERE plan_id = 'monthly1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_purchasers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'monthly1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_purchasers_returning_from_monthly_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_purchasers_returning_from_monthly_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_purchasers_returning_from_monthly_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_purchasers_returning_from_monthly_cats
),

gift_purchasers_returning_from_annual_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_purchasers_returning WHERE plan_id = 'annual1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_purchasers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'annual1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_purchasers_returning_from_annual_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_purchasers_returning_from_annual_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_purchasers_returning_from_annual_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_purchasers_returning_from_annual_cats
),

gift_purchasers_returning_from_gift_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_purchasers_returning WHERE plan_id = 'gift-plan'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_purchasers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'gift-plan' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_purchasers_returning_from_gift_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_purchasers_returning_from_gift_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_purchasers_returning_from_gift_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_purchasers_returning_from_gift_cats
),

gift_total_redeemers_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT redeemer_id) AS cnt FROM g_redeemers
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.redeemer_id)
    FROM g_redeemers xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.redeemer_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_total_redeemers_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_total_redeemers_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_total_redeemers_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_total_redeemers_cats
),

gift_redeemers_net_new_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_redeemers_net_new
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_redeemers_net_new xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_redeemers_net_new_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_redeemers_net_new_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_redeemers_net_new_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_redeemers_net_new_cats
),

gift_redeemers_returning_from_monthly_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_redeemers_returning WHERE plan_id = 'monthly1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_redeemers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'monthly1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_redeemers_returning_from_monthly_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_redeemers_returning_from_monthly_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_redeemers_returning_from_monthly_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_redeemers_returning_from_monthly_cats
),

gift_redeemers_returning_from_annual_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_redeemers_returning WHERE plan_id = 'annual1'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_redeemers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'annual1' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_redeemers_returning_from_annual_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_redeemers_returning_from_annual_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_redeemers_returning_from_annual_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_redeemers_returning_from_annual_cats
),

gift_redeemers_returning_from_gift_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM gift_redeemers_returning WHERE plan_id = 'gift-plan'
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM gift_redeemers_returning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE xm.plan_id = 'gift-plan' AND mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_redeemers_returning_from_gift_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_redeemers_returning_from_gift_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_redeemers_returning_from_gift_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_redeemers_returning_from_gift_cats
),

gift_lapsed_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM g_lapsed
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM g_lapsed xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_lapsed_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_lapsed_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_lapsed_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_lapsed_cats
),

monthly_active_balance_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM bal_monthly_active
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM bal_monthly_active xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_active_balance_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_active_balance_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_active_balance_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_active_balance_cats
),

annual_active_balance_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM bal_annual_active
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM bal_annual_active xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
annual_active_balance_by_rfm AS (
    SELECT rfm_category, cnt FROM annual_active_balance_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM annual_active_balance_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM annual_active_balance_cats
),

gift_active_balance_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM bal_gift_active
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM bal_gift_active xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
gift_active_balance_by_rfm AS (
    SELECT rfm_category, cnt FROM gift_active_balance_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM gift_active_balance_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM gift_active_balance_cats
),

monthly_members_dunning_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM bal_monthly_dunning
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM bal_monthly_dunning xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
monthly_members_dunning_by_rfm AS (
    SELECT rfm_category, cnt FROM monthly_members_dunning_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM monthly_members_dunning_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM monthly_members_dunning_cats
),

-- total_active_balance: union of monthly active + annual active + gift
-- active + monthly dunning + annual dunning (a_eom_dunning included —
-- this was missing in the prior draft and undercounted the total).
bal_all_active_union AS (
    SELECT cc_user_profile_id FROM bal_monthly_active
    UNION
    SELECT cc_user_profile_id FROM bal_annual_active
    UNION
    SELECT cc_user_profile_id FROM bal_gift_active
    UNION
    SELECT cc_user_profile_id FROM bal_monthly_dunning
    UNION
    SELECT cc_user_profile_id FROM a_eom_dunning
),
total_active_balance_cats AS (
    SELECT 'All' AS rfm_category, COUNT(DISTINCT cc_user_profile_id) AS cnt FROM bal_all_active_union
    UNION ALL
    SELECT mcm.rfm_category, COUNT(DISTINCT xm.cc_user_profile_id)
    FROM bal_all_active_union xm JOIN member_category_map mcm ON mcm.cc_user_profile_id = xm.cc_user_profile_id
    WHERE mcm.rfm_category != 'Dormant'
    GROUP BY mcm.rfm_category
),
total_active_balance_by_rfm AS (
    SELECT rfm_category, cnt FROM total_active_balance_cats
    UNION ALL
    SELECT 'Top 3 RFM', SUM(cnt) FROM total_active_balance_cats WHERE rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals')
    UNION ALL
    SELECT 'Dormant', MAX(CASE WHEN rfm_category='All' THEN cnt ELSE 0 END) - SUM(CASE WHEN rfm_category!='All' THEN cnt ELSE 0 END) FROM total_active_balance_cats
)

-- =====================================================================
-- FINAL SELECT — one LEFT JOIN per metric, COALESCE to 0 for any
-- category a metric had no rows for.
-- =====================================================================
SELECT
    (SELECT month_start FROM report_period) AS month_start,
    (SELECT month_end FROM report_period) AS month_end,
    c.rfm_category,

    -- Monthly
    COALESCE(ms.cnt, 0) AS monthly_signups,
    COALESCE(mft.cnt, 0) AS monthly_first_time_signups,
    COALESCE(mr.cnt, 0) AS monthly_returning_signups,
    COALESCE(mrm.cnt, 0) AS monthly_returning_from_monthly,
    COALESCE(mra.cnt, 0) AS monthly_returning_from_annual,
    COALESCE(mrg.cnt, 0) AS monthly_returning_from_gift,
    COALESCE(mex.cnt, 0) AS monthly_exits,
    COALESCE(ms.cnt, 0) - COALESCE(mex.cnt, 0) AS monthly_net_increase,

    -- Annual
    COALESCE(as_.cnt, 0) AS annual_signups,
    COALESCE(aft.cnt, 0) AS annual_first_time_signups,
    COALESCE(ar.cnt, 0) AS annual_returning_signups,
    COALESCE(arm.cnt, 0) AS annual_returning_from_monthly,
    COALESCE(ara.cnt, 0) AS annual_returning_from_annual,
    COALESCE(arg.cnt, 0) AS annual_returning_from_gift,
    COALESCE(ui.cnt, 0) AS upgrades_initiated,
    COALESCE(ute_.cnt, 0) AS upgrades_took_effect,
    COALESCE(aex.cnt, 0) AS annual_exits,
    COALESCE(aren.cnt, 0) AS annual_renewed,
    COALESCE(adun.cnt, 0) AS annual_dunning,
    COALESCE(alap.cnt, 0) AS annual_lapsed,
    COALESCE(as_.cnt, 0) - COALESCE(aex.cnt, 0) AS annual_net_increase,

    -- Gift
    COALESCE(gtp.cnt, 0) AS gift_total_purchasers,
    COALESCE(gpnn.cnt, 0) AS gift_purchasers_net_new,
    COALESCE(gprm.cnt, 0) AS gift_purchasers_returning_from_monthly,
    COALESCE(gpra.cnt, 0) AS gift_purchasers_returning_from_annual,
    COALESCE(gprg.cnt, 0) AS gift_purchasers_returning_from_gift,

    COALESCE(gtr.cnt, 0) AS gift_total_redeemers,
    COALESCE(grnn.cnt, 0) AS gift_redeemers_net_new,
    COALESCE(grrm.cnt, 0) AS gift_redeemers_returning_from_monthly,
    COALESCE(grra.cnt, 0) AS gift_redeemers_returning_from_annual,
    COALESCE(grrg.cnt, 0) AS gift_redeemers_returning_from_gift,

    COALESCE(glap.cnt, 0) AS gift_lapsed,

    -- Active balance (as of EOM)
    COALESCE(mab.cnt, 0) AS monthly_active_balance,
    COALESCE(aab.cnt, 0) AS annual_active_balance,
    COALESCE(gab.cnt, 0) AS gift_active_balance,
    COALESCE(mdun.cnt, 0) AS monthly_members_dunning,
    COALESCE(tab.cnt, 0) AS total_active_balance

FROM categories c
LEFT JOIN monthly_signups_by_rfm ms ON ms.rfm_category = c.rfm_category
LEFT JOIN monthly_first_time_by_rfm mft ON mft.rfm_category = c.rfm_category
LEFT JOIN monthly_returning_by_rfm mr ON mr.rfm_category = c.rfm_category
LEFT JOIN monthly_returning_from_monthly_by_rfm mrm ON mrm.rfm_category = c.rfm_category
LEFT JOIN monthly_returning_from_annual_by_rfm mra ON mra.rfm_category = c.rfm_category
LEFT JOIN monthly_returning_from_gift_by_rfm mrg ON mrg.rfm_category = c.rfm_category
LEFT JOIN monthly_exits_by_rfm mex ON mex.rfm_category = c.rfm_category
LEFT JOIN annual_signups_by_rfm as_ ON as_.rfm_category = c.rfm_category
LEFT JOIN annual_first_time_by_rfm aft ON aft.rfm_category = c.rfm_category
LEFT JOIN annual_returning_by_rfm ar ON ar.rfm_category = c.rfm_category
LEFT JOIN annual_returning_from_monthly_by_rfm arm ON arm.rfm_category = c.rfm_category
LEFT JOIN annual_returning_from_annual_by_rfm ara ON ara.rfm_category = c.rfm_category
LEFT JOIN annual_returning_from_gift_by_rfm arg ON arg.rfm_category = c.rfm_category
LEFT JOIN upgrades_initiated_by_rfm ui ON ui.rfm_category = c.rfm_category
LEFT JOIN upgrades_took_effect_by_rfm ute_ ON ute_.rfm_category = c.rfm_category
LEFT JOIN annual_exits_by_rfm aex ON aex.rfm_category = c.rfm_category
LEFT JOIN annual_renewed_by_rfm aren ON aren.rfm_category = c.rfm_category
LEFT JOIN annual_dunning_by_rfm adun ON adun.rfm_category = c.rfm_category
LEFT JOIN annual_lapsed_by_rfm alap ON alap.rfm_category = c.rfm_category
LEFT JOIN gift_total_purchasers_by_rfm gtp ON gtp.rfm_category = c.rfm_category
LEFT JOIN gift_purchasers_net_new_by_rfm gpnn ON gpnn.rfm_category = c.rfm_category
LEFT JOIN gift_purchasers_returning_from_monthly_by_rfm gprm ON gprm.rfm_category = c.rfm_category
LEFT JOIN gift_purchasers_returning_from_annual_by_rfm gpra ON gpra.rfm_category = c.rfm_category
LEFT JOIN gift_purchasers_returning_from_gift_by_rfm gprg ON gprg.rfm_category = c.rfm_category
LEFT JOIN gift_total_redeemers_by_rfm gtr ON gtr.rfm_category = c.rfm_category
LEFT JOIN gift_redeemers_net_new_by_rfm grnn ON grnn.rfm_category = c.rfm_category
LEFT JOIN gift_redeemers_returning_from_monthly_by_rfm grrm ON grrm.rfm_category = c.rfm_category
LEFT JOIN gift_redeemers_returning_from_annual_by_rfm grra ON grra.rfm_category = c.rfm_category
LEFT JOIN gift_redeemers_returning_from_gift_by_rfm grrg ON grrg.rfm_category = c.rfm_category
LEFT JOIN gift_lapsed_by_rfm glap ON glap.rfm_category = c.rfm_category
LEFT JOIN monthly_active_balance_by_rfm mab ON mab.rfm_category = c.rfm_category
LEFT JOIN annual_active_balance_by_rfm aab ON aab.rfm_category = c.rfm_category
LEFT JOIN gift_active_balance_by_rfm gab ON gab.rfm_category = c.rfm_category
LEFT JOIN monthly_members_dunning_by_rfm mdun ON mdun.rfm_category = c.rfm_category
LEFT JOIN total_active_balance_by_rfm tab ON tab.rfm_category = c.rfm_category

ORDER BY
    CASE c.rfm_category
        WHEN 'All' THEN 0
        WHEN 'Top 3 RFM' THEN 1
        WHEN 'Champions' THEN 2
        WHEN 'Potential Loyalist' THEN 3
        WHEN 'Value Conscious Loyals' THEN 4
        WHEN 'Promising' THEN 5
        WHEN 'At Risk' THEN 6
        WHEN 'New' THEN 7
        WHEN 'Fleeting' THEN 8
        WHEN 'High Value Lapsed' THEN 9
        WHEN 'Dormant' THEN 10
    END